### Household Rosters
### _hhr.ipynb


Sarah Sullivan

Created: April 7, 2026 

Last Updated: May 19, 2026


This script inputs the file "_psid_long_lean.dta," outputted by the script _psid.do in part I, step 25. 
I do some datatype manipulation to create variables measuring changes between household rosters constructed in _psid.do. 
I then output the csv file "_hhr.csv" which gets merged back onto the file "_psid_long.dta" in part X, step X of _psid.do. 

In [2]:
# import packages
# version of pandas is 2.0.3, numpy is 2.0.3
import numpy as np
import pandas as pd

In [3]:
# read in data output from _psid.do part I step 25. 
output = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/_output"
df = pd.read_stata(output + "/_psid_long_lean.dta", convert_categoricals=False)

In [4]:
# convert float variables to integers
floatvars = ["fam", "ID", "age_", "fam_id_", "sample_indiv_N", "sample_indiv_A", "sample_indiv_B"]

for i in floatvars:
    df.fillna({i:0}, inplace=True)
    df[i] = df[i].astype(int)

In [5]:
# convert list-like strings to real lists

df['hhr'] = [[] for _ in range(len(df))]
df['ages_hhr'] = [[] for _ in range(len(df))]
df['rel_hhr'] = [[] for _ in range(len(df))]
df['siblings'] = [[] for _ in range(len(df))]
df['parents'] = [[] for _ in range(len(df))]
df['grandparents'] = [[] for _ in range(len(df))]

df['hhr'] = df["hhr_no_self"].apply(lambda x: x.split())
df['ages_hhr'] = df["ages_no_self"].apply(lambda x: x.split())
df['rel_hhr'] = df["rel_no_self"].apply(lambda x: x.split())
df['siblings'] = df["sib_list"].apply(lambda x: x.split())
df['parents'] = df["par_list"].apply(lambda x: x.split())
df['grandparents'] = df["gpar_list"].apply(lambda x: x.split())

In [6]:
# clean her up 
df = df.drop(columns=['hhr_no_self', 'ages_no_self', 'rel_no_self', 'sib_list', 'par_list', 'gpar_list'])

In [7]:
# within-person previous roster (ordered by year)
df = df.sort_values(["ID", "year"]).copy()

df["hhr_prev"] = df.groupby("ID")["hhr"].shift(1)
df["ages_prev"] = df.groupby("ID")["ages_hhr"].shift(1)
df["rel_prev"] = df.groupby("ID")["rel_hhr"].shift(1)

In [8]:
# fill missings with empty lists
df['hhr_prev'] = df['hhr_prev'].apply(lambda d: d if isinstance(d, list) else [])
df['ages_prev'] = df['ages_prev'].apply(lambda d: d if isinstance(d, list) else [])
df['rel_prev'] = df['rel_prev'].apply(lambda d: d if isinstance(d, list) else [])

In [9]:
# ids of who left and who came in each year (within person)
df["IDs_left"] = df.apply(
    lambda row: [x for x in row["hhr_prev"] if x not in row["hhr"]],
    axis=1
)
df["IDs_came"] = df.apply(
    lambda row: [x for x in row["hhr"] if x not in row["hhr_prev"]],
    axis=1
)

In [10]:
# flag first and last year of observation for each individual
df['is_first'] = ~df['ID'].duplicated(keep='first')
df['is_last'] = ~df['ID'].duplicated(keep='last')

In [11]:
df['IDs_came'] = df.apply(lambda row: [] if row['is_first'] else row['IDs_came'], axis=1)
df['IDs_left'] = df.apply(lambda row: [] if row['is_last'] else row['IDs_left'], axis=1)

In [12]:
# function for age of who left/came

def get_ages_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    IDs_left = row['IDs_left'] if isinstance(row['IDs_left'], list) else []
    return [ages_prev[hhr_prev.index(pid)] for pid in IDs_left if pid in hhr_prev and hhr_prev.index(pid) < len(ages_prev)]

def get_ages_came(row):
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    ages = row['ages_hhr'] if isinstance(row['ages_hhr'], list) else []
    IDs_came = row['IDs_came'] if isinstance(row['IDs_came'], list) else []
    return [ages[hhr.index(pid)] for pid in IDs_came if pid in hhr and hhr.index(pid) < len(ages)]  

In [13]:
# function for relationship of who left to head

def get_rel_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    rel_prev = row['rel_prev'] if isinstance(row['rel_prev'], list) else []
    IDs_left = row['IDs_left'] if isinstance(row['IDs_left'], list) else []
    return [rel_prev[hhr_prev.index(pid)] for pid in IDs_left if pid in hhr_prev and hhr_prev.index(pid) < len(rel_prev)]

def get_rel_came(row):
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    rel = row['rel_hhr'] if isinstance(row['rel_hhr'], list) else []
    IDs_came = row['IDs_came'] if isinstance(row['IDs_came'], list) else []
    return [rel[hhr.index(pid)] for pid in IDs_came if pid in hhr and hhr.index(pid) < len(rel)]  

In [14]:
# apply fns above 

df['ages_left'] = df.apply(get_ages_left, axis=1)
df['ages_came'] = df.apply(get_ages_came, axis=1)

df['rel_left'] = df.apply(get_rel_left, axis=1)
df['rel_came'] = df.apply(get_rel_came, axis=1)

In [15]:
# convert elements of ages_left and ages_came to list of integers
df['ages_left'] = df['ages_left'].apply(lambda ages: ([int(age) for age in ages]) if isinstance(ages, list) else [])
df['ages_came'] = df['ages_came'].apply(lambda ages: ([int(age) for age in ages]) if isinstance(ages, list) else [])

In [16]:
# sort by adult vs. child came or left. 

df['adult_came'] = df['ages_came'].apply(
    lambda ages: isinstance(ages, list) and any(age >= 18 for age in ages)
)

df['child_came'] = df['ages_came'].apply(
    lambda ages: isinstance(ages, list) and any(age < 18 for age in ages)
)

df['adult_left'] = df['ages_left'].apply(
    lambda ages: isinstance(ages, list) and any(age >= 18 for age in ages)
)

df['child_left'] = df['ages_left'].apply(
    lambda ages: isinstance(ages, list) and any(age < 18 for age in ages)
)

In [17]:
df['sib_came'] = df.apply(
    lambda row: (isinstance(row['IDs_came'], list) and isinstance(row['siblings'], list) and any(id in row['siblings'] for id in row['IDs_came'])), axis=1
    )

df['sib_left'] = df.apply(
    lambda row: (isinstance(row['IDs_left'], list) and isinstance(row['siblings'], list) and any(id in row['siblings'] for id in row['IDs_left'])), axis=1
)   

In [18]:
# Get ages of siblings who came
def get_sib_ages_came(row):
    if not isinstance(row['IDs_came'], list) or not isinstance(row['siblings'], list):
        return []
    hhr = row['hhr'] if isinstance(row['hhr'], list) else []
    ages = row['ages_hhr'] if isinstance(row['ages_hhr'], list) else []
    # Get IDs that are both in IDs_came and siblings
    sibs_IDs_came = [id for id in row['IDs_came'] if id in row['siblings']]
    # Get ages for those siblings
    return [ages[hhr.index(sib_id)] for sib_id in sibs_IDs_came if sib_id in hhr and hhr.index(sib_id) < len(ages)]

# Get ages of siblings who left
def get_sib_ages_left(row):
    if not isinstance(row['IDs_left'], list) or not isinstance(row['siblings'], list):
        return []
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    # Get IDs that are both in IDs_left and siblings
    sibs_IDs_left = [id for id in row['IDs_left'] if id in row['siblings']]
    # Get ages for those siblings
    return [ages_prev[hhr_prev.index(sib_id)] for sib_id in sibs_IDs_left if sib_id in hhr_prev and hhr_prev.index(sib_id) < len(ages_prev)]

df['sib_ages_came'] = df.apply(get_sib_ages_came, axis=1)
df['sib_ages_left'] = df.apply(get_sib_ages_left, axis=1)

In [19]:
df['par_came'] = df.apply(
    lambda row: int(isinstance(row['IDs_came'], list) and isinstance(row['parents'], list) and any(id in row['parents'] for id in row['IDs_came'])),
    axis=1
)

df['par_left'] = df.apply(
    lambda row: int(isinstance(row['IDs_left'], list) and isinstance(row['parents'], list) and any(id in row['parents'] for id in row['IDs_left'])),
    axis=1
)

In [20]:
df['gpar_came'] = df.apply(
    lambda row: int(isinstance(row['IDs_came'], list) and isinstance(row['grandparents'], list) and any(id in row['grandparents'] for id in row['IDs_came'])),
    axis=1
)

df['gpar_left'] = df.apply(
    lambda row: int(isinstance(row['IDs_left'], list) and isinstance(row['grandparents'], list) and any(id in row['grandparents'] for id in row['IDs_left'])),
    axis=1
)

In [21]:
# flags for changes, in, and out. 
df['hhr_change'] = df.apply(lambda row: 1 if (len(row['IDs_came']) > 0) or (len(row['IDs_left']) > 0) else 0, axis=1)
df['hhr_in'] = df.apply(lambda row: 1 if len(row['IDs_came']) > 0 else 0, axis=1)
df['hhr_out'] = df.apply(lambda row: 1 if len(row['IDs_left']) > 0 else 0, axis=1)

In [22]:
# output
df.to_csv(f"{output}/_hhr.csv", index=False)

Graveyard

In [23]:
"""

df2["birth_year"] = (df2["yr"] - df2["age_"]).astype(int)
df2["birth_year"] = df2.groupby("ID")["birth_year"].transform("min").astype(int)

start_year = 1948

# 5-yr cohorts
bin_size = 5

# Create bins from 1948 to just past 2020
bins_5 = range(start_year, 2026, bin_size)

# Create labels like "1948-1953", "1954-1959", etc.
labels_5 = [f"{y}-{y + bin_size - 1}" for y in bins_5[:-1]]

# Assign 5-year cohorts
df2['cohort_5'] = pd.cut(
    df2['birth_year'],
    bins=list(bins_5),
    labels=labels_5,
    right=False  # intervals are [left, right), so 1948 <= x < 1953
)

# 10-yr cohorts
bin_size = 10

# Create bins from 1948 to just past 2020
bins_10 = range(start_year, 2026, bin_size)

# Create labels like "1948-1957", "1958-1967", etc.
labels_10 = [f"{y}-{y + bin_size - 1}" for y in bins_10[:-1]]

# Assign 10-year cohorts
df2['cohort_10'] = pd.cut(
    df2['birth_year'],
    bins=list(bins_10),
    labels=labels_10,
    right=False  # intervals are [left, right), so 1948 <= x < 1958
)

"""

'\n\ndf2["birth_year"] = (df2["yr"] - df2["age_"]).astype(int)\ndf2["birth_year"] = df2.groupby("ID")["birth_year"].transform("min").astype(int)\n\nstart_year = 1948\n\n# 5-yr cohorts\nbin_size = 5\n\n# Create bins from 1948 to just past 2020\nbins_5 = range(start_year, 2026, bin_size)\n\n# Create labels like "1948-1953", "1954-1959", etc.\nlabels_5 = [f"{y}-{y + bin_size - 1}" for y in bins_5[:-1]]\n\n# Assign 5-year cohorts\ndf2[\'cohort_5\'] = pd.cut(\n    df2[\'birth_year\'],\n    bins=list(bins_5),\n    labels=labels_5,\n    right=False  # intervals are [left, right), so 1948 <= x < 1953\n)\n\n# 10-yr cohorts\nbin_size = 10\n\n# Create bins from 1948 to just past 2020\nbins_10 = range(start_year, 2026, bin_size)\n\n# Create labels like "1948-1957", "1958-1967", etc.\nlabels_10 = [f"{y}-{y + bin_size - 1}" for y in bins_10[:-1]]\n\n# Assign 10-year cohorts\ndf2[\'cohort_10\'] = pd.cut(\n    df2[\'birth_year\'],\n    bins=list(bins_10),\n    labels=labels_10,\n    right=False  # int